In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql.window import *

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@databricksgvse2e.dfs.core.windows.net/orders")
display(df)

In [0]:
df =df.drop("_rescued_data")

In [0]:
df_date = df.withColumn("order_date", to_timestamp(col("order_date")))
df_year =df_date.withColumn("year", year(col("order_date")))
display(df_year)


In [0]:
class windows:
    def rank(self, df, partition, order, column):
        return df.withColumn(column, rank().over(Window.partitionBy(partition).orderBy(desc(order))))
    def dense_rank1(self,df,partition,order,column):
                return df.withColumn(column, dense_rank().over(Window.partitionBy(partition).orderBy(desc(order))))
    def row_number(self,df,partition,order,column):
                return df.withColumn(column, row_number().over(Window.partitionBy(partition).orderBy(desc(order))))

In [0]:
#Applying Ranks on year.

w = windows()
df_year_rank = w.rank(df_year, "year", "total_amount", "rank")
df_year_dense_rank =  w.dense_rank1(df_year_rank, "year", "total_amount", "dense_rank")
df_year_final = w.row_number(df_year_dense_rank, "year", "total_amount", "row_number")
display(df_year_final)

In [0]:
df_year_final.write.mode("overwrite").format("delta").save("abfss://silver@databricksgvse2e.dfs.core.windows.net/orders")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_cat.silver.orders
USING DELTA
LOCATION 'abfss://silver@databricksgvse2e.dfs.core.windows.net/orders'
